In [61]:
# Import python modules
import os
import sys
import pandas as pd
import numpy as np

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)

In [62]:
# Import custom modules
import plotting
import utils

In [63]:
folder = os.path.join(os.path.dirname(os.getcwd()), "configs")

In [64]:
path = os.path.join(folder, "models", "config_37_v1_multi_mini.yaml")
if not utils.load_model_config()["years"]:
    print("Yee")

Path_or_name: 
Configuration loaded from c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\base_config.yaml
Yee


In [65]:
data_folder_path = os.path.join(
    os.path.dirname(os.getcwd()), "data", "processed", "elec_s_37_ES_PT_no_bat_limit"
)
data_folder_path

'c:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\data\\processed\\elec_s_37_ES_PT_no_bat_limit'

In [ ]:
def test_function(
    years: list[int], weeks, data_folder_path: str
) -> dict[str, pd.DataFrame]:
    """temporary quick fix for multi-year data loading. I use the same data as for a single year, but I just post-process the dataframes to be in a multi-year format. All data is the same accross all years, so results are relatively meaningless."""
    if not os.path.exists(data_folder_path):
        raise FileNotFoundError(
            f"{data_folder_path} not found (should be the path to a folder containing processed data in csv files)"
        )
    data = {}
    for file in os.listdir(data_folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(data_folder_path, file)
            file_name = file.split(".")[0]
            if file_name in ["hourly_demand", "capacity_factors"]:
                df = pd.read_csv(file_path, index_col=0, parse_dates=True)
                df["week"] = df.index.isocalendar().week
                df["month"] = df.index.month
                df["hour"] = df.index.hour
                hours = df["hour"].unique()
            else:
                df = pd.read_csv(file_path, index_col=0)
            new_dfs = []
            demand_multiplier = 1
            if file_name == "nodes":
                data[file_name] = df
                continue
            for year in years:
                temp_df = df.copy()
                if file_name == "hourly_demand":
                    temp_df = temp_df * demand_multiplier
                    demand_multiplier += 1
                if file_name in ["hourly_demand", "capacity_factors"]:
                    temp_df.index = pd.MultiIndex.from_product(
                        [[year], weeks, hours],
                        names=["year", "week", "hour"],
                    )
                else:
                    temp_df.index = pd.MultiIndex.from_product(
                        [[year], temp_df.index], names=["year", temp_df.index.name]
                    )
                new_dfs.append(temp_df)
            data[file_name] = pd.concat(new_dfs)
    return data

In [105]:
years = [2025, 2035, 2045]
weeks = [3, 16, 29, 42]

In [106]:
input_data = test_function(years, weeks, data_folder_path)

TypeError: Input must be list-like

In [88]:
input_data["batteries"]

node  MC  capital_cost  hour_capacity  cdrate  \
year [battery]                                                   
2025 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   
2035 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   
2045 ES1 0 bat  ES1 0   0   21958.92494              2     1.0   
     PT1 0 bat  PT1 0   0   21958.92494              2     1.0   

                P_discharge_max  P_discharge_min  P_charge_max  P_charge_min  \
year [battery]                                                                 
2025 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   
2035 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   
2045 ES1 0 bat        100000000                0   100000000.0             0   
     PT1 0 bat        100000000                0   100000000.0             0   

                SOC_max  SOC_min     delta  eta_charge  eta_discharge  
year [battery]                                                         
2025 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97  
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97  
2035 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97  
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97  
2045 ES1 0 bat      0.9      0.1  0.000042        0.98           0.97  
     PT1 0 bat      0.9      0.1  0.000042        0.98           0.97

In [89]:
input_data["capacity_factors"]

ES1 0 offwind-ac  ES1 0 onwind  ES1 0 ror  \
year [snapshot]                                                       
2025 2013-01-01 00:00:00          0.169080      0.198307   0.223631   
     2013-01-01 01:00:00          0.180396      0.183442   0.209677   
     2013-01-01 02:00:00          0.187817      0.173403   0.198885   
     2013-01-01 03:00:00          0.198432      0.160254   0.187313   
     2013-01-01 04:00:00          0.218048      0.156526   0.179705   
...                                    ...           ...        ...   
2045 2013-12-31 19:00:00          0.177343      0.219160   0.300193   
     2013-12-31 20:00:00          0.181897      0.228008   0.288119   
     2013-12-31 21:00:00          0.190584      0.231160   0.279827   
     2013-12-31 22:00:00          0.244625      0.273985   0.273984   
     2013-12-31 23:00:00          0.259790      0.272601   0.269047   

                          ES1 0 solar  PT1 0 offwind-ac  PT1 0 onwind  \
year [snapshot]                                                         
2025 2013-01-01 00:00:00          0.0          0.223462      0.140790   
     2013-01-01 01:00:00          0.0          0.242486      0.109833   
     2013-01-01 02:00:00          0.0          0.251477      0.101679   
     2013-01-01 03:00:00          0.0          0.246618      0.098385   
     2013-01-01 04:00:00          0.0          0.247457      0.103719   
...                               ...               ...           ...   
2045 2013-12-31 19:00:00          0.0          0.196510      0.116910   
     2013-12-31 20:00:00          0.0          0.187475      0.109712   
     2013-12-31 21:00:00          0.0          0.172794      0.102505   
     2013-12-31 22:00:00          0.0          0.238736      0.138531   
     2013-12-31 23:00:00          0.0          0.191602      0.125415   

                          PT1 0 ror  PT1 0 solar  ES1 0 CCGT  PT1 0 CCGT  \
year [snapshot]                                                            
2025 2013-01-01 00:00:00   0.108399          0.0         1.0         1.0   
     2013-01-01 01:00:00   0.104136          0.0         1.0         1.0   
     2013-01-01 02:00:00   0.100124          0.0         1.0         1.0   
     2013-01-01 03:00:00   0.094812          0.0         1.0         1.0   
     2013-01-01 04:00:00   0.089900          0.0         1.0         1.0   
...                             ...          ...         ...         ...   
2045 2013-12-31 19:00:00   0.636055          0.0         1.0         1.0   
     2013-12-31 20:00:00   0.621952          0.0         1.0         1.0   
     2013-12-31 21:00:00   0.611703          0.0         1.0         1.0   
     2013-12-31 22:00:00   0.601698          0.0         1.0         1.0   
     2013-12-31 23:00:00   0.594096          0.0         1.0         1.0   

                          ES1 0 coal  week  month  hour  
year [snapshot]                                          
2025 2013-01-01 00:00:00         1.0     1      1     0  
     2013-01-01 01:00:00         1.0     1      1     1  
     2013-01-01 02:00:00         1.0     1      1     2  
     2013-01-01 03:00:00         1.0     1      1     3  
     2013-01-01 04:00:00         1.0     1      1     4  
...                              ...   ...    ...   ...  
2045 2013-12-31 19:00:00         1.0     1     12    19  
     2013-12-31 20:00:00         1.0     1     12    20  
     2013-12-31 21:00:00         1.0     1     12    21  
     2013-12-31 22:00:00         1.0     1     12    22  
     2013-12-31 23:00:00         1.0     1     12    23  

[26280 rows x 14 columns]

In [91]:
batteries = pd.read_csv(os.path.join(data_folder_path, "batteries.csv"), index_col=0)
batteries

,node,MC,capital_cost,hour_capacity,cdrate,P_discharge_max,P_discharge_min,P_charge_max,P_charge_min,SOC_max,SOC_min,delta,eta_charge,eta_discharge
battery,,,,,,,,,,,,,,
ES1 0 bat,ES1 0,0,21958.92494,2,1.0,100000000,0,100000000.0,0,0.9,0.1,0.000042,0.98,0.97
PT1 0 bat,PT1 0,0,21958.92494,2,1.0,100000000,0,100000000.0,0,0.9,0.1,0.000042,0.98,0.97


In [ ]:
capacity_factors = pd.read_csv(
    os.path.join(data_folder_path, "capacity_factors.csv"), index_col=0
)
capacity_factors.index = capacity_factors.index.astype(dtype="datetime64[ns]")
capacity_factors["week"] = capacity_factors.index.to_series().dt.isocalendar().week
capacity_factors["month"] = capacity_factors.index.to_series().dt.month
capacity_factors["hour"] = capacity_factors.index.hour

In [ ]:
capacity_factors

,ES1 0 offwind-ac,ES1 0 onwind,ES1 0 ror,ES1 0 solar,PT1 0 offwind-ac,PT1 0 onwind,PT1 0 ror,PT1 0 solar,ES1 0 CCGT,PT1 0 CCGT,ES1 0 coal,week,month,hour
snapshot,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.169080,0.198307,0.223631,0.0,0.223462,0.140790,0.108399,0.0,1.0,1.0,1.0,1,1,0
2013-01-01 01:00:00,0.180396,0.183442,0.209677,0.0,0.242486,0.109833,0.104136,0.0,1.0,1.0,1.0,1,1,1
2013-01-01 02:00:00,0.187817,0.173403,0.198885,0.0,0.251477,0.101679,0.100124,0.0,1.0,1.0,1.0,1,1,2
2013-01-01 03:00:00,0.198432,0.160254,0.187313,0.0,0.246618,0.098385,0.094812,0.0,1.0,1.0,1.0,1,1,3
2013-01-01 04:00:00,0.218048,0.156526,0.179705,0.0,0.247457,0.103719,0.089900,0.0,1.0,1.0,1.0,1,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-31 19:00:00,0.177343,0.219160,0.300193,0.0,0.196510,0.116910,0.636055,0.0,1.0,1.0,1.0,1,12,19
2013-12-31 20:00:00,0.181897,0.228008,0.288119,0.0,0.187475,0.109712,0.621952,0.0,1.0,1.0,1.0,1,12,20
2013-12-31 21:00:00,0.190584,0.231160,0.279827,0.0,0.172794,0.102505,0.611703,0.0,1.0,1.0,1.0,1,12,21


In [94]:
# Load the dataframe
df = pd.read_csv(
    os.path.join(data_folder_path, "capacity_factors.csv"),
    index_col=0,
    parse_dates=True,
)
# Extract (year, week, hour) from the datetime index
df.index = pd.to_datetime(df.index)
multi_index = pd.MultiIndex.from_arrays(
    [df.index.year, df.index.isocalendar().week, df.index.hour],
    names=["year", "week", "hour"],
)
# Assign new multi-index to dataframe
df.index = multi_index

In [97]:
df.index

MultiIndex([(2013, 1,  0),
            (2013, 1,  1),
            (2013, 1,  2),
            (2013, 1,  3),
            (2013, 1,  4),
            (2013, 1,  5),
            (2013, 1,  6),
            (2013, 1,  7),
            (2013, 1,  8),
            (2013, 1,  9),
            ...
            (2013, 1, 14),
            (2013, 1, 15),
            (2013, 1, 16),
            (2013, 1, 17),
            (2013, 1, 18),
            (2013, 1, 19),
            (2013, 1, 20),
            (2013, 1, 21),
            (2013, 1, 22),
            (2013, 1, 23)],
           names=['year', 'week', 'hour'], length=8760)